In [3]:
import os
import pandas as pd
import glob

def merge_and_analyze():
    # 1. Load Data
    path = r'Csv_folders\Csv_folders'
    all_files = glob.glob(os.path.join(path, "*.csv"))
    
    if not all_files:
        print(f"No CSV files found in {path}")
        return

    print(f"Found {len(all_files)} CSV files.")
    
    df_list = []
    for filename in all_files:
        try:
            df = pd.read_csv(filename, index_col=None, header=0)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    if not df_list:
        print("No valid dataframes to merge.")
        return

    # 2. Merge
    merged_df = pd.concat(df_list, axis=0, ignore_index=True)
    
    # 3. Clean/Preprocess
    # Ensure is_ml_origin is boolean
    if 'is_ml_origin' in merged_df.columns:
        merged_df['is_ml_origin'] = merged_df['is_ml_origin'].astype(str).str.lower() == 'true'
    else:
        merged_df['is_ml_origin'] = False # Default if missing, though it shouldn't be based on inspection

    # 4. Analyze
    total_rows = len(merged_df)
    unique_repos = merged_df['repo_name'].nunique() if 'repo_name' in merged_df.columns else 0
    
    ml_bug_count = merged_df['is_ml_origin'].sum()
    ml_bug_percent = (ml_bug_count / total_rows) * 100 if total_rows > 0 else 0
    
    # SZZ Efficacy: rows where bug_introducing_commit is present (not null/empty)
    szz_identified = merged_df['bug_introducing_commit'].notna() & (merged_df['bug_introducing_commit'] != '')
    szz_count = szz_identified.sum()
    szz_percent = (szz_count / total_rows) * 100 if total_rows > 0 else 0

    # 5. Output
    output_path = 'merged_ml_bug_analysis.csv'
    merged_df.to_csv(output_path, index=False)
    
    report_lines = []
    report_lines.append("-" * 30)
    report_lines.append("ANALYSIS REPORT")
    report_lines.append("-" * 30)
    report_lines.append(f"Total Records Processed: {total_rows}")
    report_lines.append(f"Unique Repositories: {unique_repos}")
    report_lines.append(f"\nML Origin Bugs: {ml_bug_count} ({ml_bug_percent:.2f}%)")
    report_lines.append(f"SZZ Bug-Inducing Commits Found: {szz_count} ({szz_percent:.2f}%)")

    # Repo-level stats
    if 'repo_name' in merged_df.columns:
        report_lines.append("-" * 30)
        report_lines.append("Repository Breakdown (Top 10)")
        repo_stats = merged_df.groupby('repo_name').agg(
            total=('commit_hash', 'count'),
            ml_bugs=('is_ml_origin', 'sum')
        ).sort_values('total', ascending=False)
        report_lines.append(repo_stats.head(10).to_string())

    # File Type Analysis
    if 'originating_file' in merged_df.columns:
        report_lines.append("-" * 30)
        report_lines.append("File Extension Analysis (Top 10 ML-Prone Types)")
        
        def get_ext(path):
            if pd.isna(path) or path == '':
                return 'unknown'
            base = os.path.basename(str(path))
            _, ext = os.path.splitext(base)
            return ext.lower() if ext else 'no_ext'

        merged_df['file_ext'] = merged_df['originating_file'].apply(get_ext)
        
        # Filter for ML bugs only to see where they happen most
        ml_bugs_df = merged_df[merged_df['is_ml_origin']]
        ext_stats = ml_bugs_df['file_ext'].value_counts().head(10)
        report_lines.append(ext_stats.to_string())
    
    report_content = "\n".join(report_lines)
    
    # Print to console
    print(report_content)
    print("-" * 30)
    print(f"\nMerged data saved to: {output_path}")
    
    # Save to file
    report_file_path = 'analysis_report.txt'
    with open(report_file_path, 'w') as f:
        f.write(report_content)
    print(f"Analysis report saved to: {report_file_path}")

if __name__ == "__main__":
    merge_and_analyze()

Found 13 CSV files.
------------------------------
ANALYSIS REPORT
------------------------------
Total Records Processed: 36145
Unique Repositories: 13

ML Origin Bugs: 1530 (4.23%)
SZZ Bug-Inducing Commits Found: 29461 (81.51%)
------------------------------
Repository Breakdown (Top 10)
                                       total  ml_bugs
repo_name                                            
freqtrade/freqtrade                     6130      231
psychopy/psychopy                       4386       42
metabrainz/listenbrainz-server          3459        7
CellProfiler/CellProfiler               3366       77
nautechsystems/nautilus_trader          3277        5
SanPen/GridCal                          2684      274
OpenBB-finance/OpenBBTerminal           2509       49
ilastik/ilastik                         2467      379
electricitymap/electricitymap-contrib   1904       34
mayan-edms/Mayan-EDMS                   1700       13
------------------------------
File Extension Analysis (Top 1